### Install dependencies

In [1]:
%%sh
pip install boto3 
pip install sagemaker
pip install torch torchvision torchaudio

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com


### Setup sagemaker session

In [77]:
import sagemaker
print(sagemaker.__version__)

sess = sagemaker.Session()  # Use the AWS region configured with the AWS CLI

# This is the SageMaker role you're already using, it will work just fine
role = 'arn:aws:iam::817539924586:role/SageMaker_FullAccess'

INFO:botocore.credentials:Found credentials in shared credentials file: ~/.aws/credentials


2.168.0


### Download data

In [78]:
import os
from torchvision import datasets

os.makedirs("./torchdata", exist_ok = True)
dataset1 = datasets.MNIST("torchdata", train=True, download=True)
dataset2 = datasets.MNIST("torchdata", train=False)


### Local example

In [85]:
# Train on local data. S3 URIs would work too.
training_input_path   = 'file://torchdata'

# Store model locally. A S3 URI would work too.
output_path           = 'file://torchmodel'

In [89]:
from sagemaker.pytorch import PyTorch

torch_estimator = PyTorch(
    entry_point='mnist_torch.py',
    role=role,
    instance_count=1, 
    instance_type='local',   # Train on the local CPU ('local_gpu' if it has a GPU)
    framework_version='1.13.1', 
    py_version='py39',
    hyperparameters={"batch-size": 16, "epochs": 1, "log-interval": 100},
    output_path=output_path
)

In [87]:
# Train! This will pull (once) the SageMaker CPU/GPU container for Torch to your local machine.
# Make sure that Docker is running and that docker-compose is installed

torch_estimator.fit({
    "training": training_input_path,
})

INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:botocore.credentials:Found credentials in shared credentials file: ~/.aws/credentials
INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker:Creating training-job with name: pytorch-training-2023-06-26-10-42-56-512
INFO:sagemaker.local.local_session:Starting training job
INFO:sagemaker.local.image:Using the long-lived AWS credentials found in session
INFO:sagemaker.local.image:docker compose file: 
networks:
  sagemaker-local:
    name: sagemaker-local
services:
  algo-1-cxswy:
    command: train
    container_name: xo0rpga4ay-algo-1-cxswy
    environment:
    - '[Masked]'
    - '[Masked]'
    - '[Masked]'
    - '[Masked]'
    image: 763104351884.dkr.ecr.eu-west-1.amazonaws.com/pytorch-training:1.13.1-cpu-py39
    networks:
      sagemaker-local:
        aliases:
        - algo-1-cxswy
    stdin_

Container xo0rpga4ay-algo-1-cxswy  Creating
Container xo0rpga4ay-algo-1-cxswy  Created
Attaching to xo0rpga4ay-algo-1-cxswy
xo0rpga4ay-algo-1-cxswy  | 2023-06-26 10:42:58,672 sagemaker-training-toolkit INFO     Imported framework sagemaker_pytorch_container.training
xo0rpga4ay-algo-1-cxswy  | 2023-06-26 10:42:58,673 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
xo0rpga4ay-algo-1-cxswy  | 2023-06-26 10:42:58,675 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
xo0rpga4ay-algo-1-cxswy  | 2023-06-26 10:42:58,683 sagemaker-training-toolkit INFO     instance_groups entry not present in resource_config
xo0rpga4ay-algo-1-cxswy  | 2023-06-26 10:42:58,686 sagemaker_pytorch_container.training INFO     Block until all host DNS lookups succeed.
xo0rpga4ay-algo-1-cxswy  | 2023-06-26 10:42:58,689 sagemaker_pytorch_container.training INFO     Invoking user training script.
xo0rpga4ay-algo-1-cxswy  | 2023-06-26 10:42:58,714 b

### AWS example

In [90]:
# upload data to S3 if required
response = sess.s3_client.list_objects(Bucket="pg-algo-data", Prefix="torchdata")
if not 'Contents' in response:
    for root,dirs,files in os.walk("torchdata"):
        for file in files:
            sess.s3_client.upload_file(os.path.join(root,file),"pg-algo-data", os.path.join(root,file))

# Train on local data. S3 URIs would work too.
training_input_path   = 's3://pg-algo-data/torchdata'

# Store model locally. A S3 URI would work too.
output_path           = 's3://pg-algo-data/torchmodel'

In [91]:
from sagemaker.pytorch import PyTorch

torch_estimator = PyTorch(
    entry_point='mnist_torch.py',
    role=role,
    instance_count=1, 
    instance_type='ml.g4dn.xlarge',   # Train on the local CPU ('local_gpu' if it has a GPU)
    framework_version='1.13.1', 
    py_version='py39',
    hyperparameters={"batch-size": 16, "epochs": 1, "log-interval": 100},
    output_path=output_path
)

INFO:botocore.credentials:Found credentials in shared credentials file: ~/.aws/credentials


In [92]:
torch_estimator.fit({
    "training": training_input_path,
})

INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.


Using provided s3_resource


INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker:Creating training-job with name: pytorch-training-2023-06-26-10-45-22-907


2023-06-26 10:45:23 Starting - Starting the training job...
2023-06-26 10:45:38 Starting - Preparing the instances for training......
2023-06-26 10:46:50 Downloading - Downloading input data...
2023-06-26 10:47:10 Training - Downloading the training image..................
2023-06-26 10:50:32 Training - Training image download completed. Training in progress...bash: cannot set terminal process group (-1): Inappropriate ioctl for device
bash: no job control in this shell
2023-06-26 10:50:43,867 sagemaker-training-toolkit INFO     Imported framework sagemaker_pytorch_container.training
2023-06-26 10:50:43,884 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2023-06-26 10:50:43,895 sagemaker_pytorch_container.training INFO     Block until all host DNS lookups succeed.
2023-06-26 10:50:43,898 sagemaker_pytorch_container.training INFO     Invoking user training script.
2023-06-26 10:50:44,145 sagemaker-training-toolkit INFO     No Neurons detected (no